In [2]:
import os
import sys
from typing import Optional, List, Dict

import pandas as pd


ANALYSIS_DIR = os.path.join("data", "analysis")
DERIVED_DIR = "data_derived"


def log(msg: str) -> None:
    print(msg, flush=True)


def pick_name_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["Name", "NAME", "neighborhood", "Neighborhood", "neighborhood_name"]:
        if c in df.columns:
            return c
    return None


def normalize_key_series(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )


def safe_read_csv(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as e:
        raise SystemExit(f"Failed to read {path}: {e}")


def suffix_columns(df: pd.DataFrame, exclude: List[str], suffix: str) -> pd.DataFrame:
    ren: Dict[str, str] = {}
    for c in df.columns:
        if c in exclude:
            continue
        ren[c] = f"{c}{suffix}"
    return df.rename(columns=ren)


def main() -> None:
    houses_csv = os.path.join(ANALYSIS_DIR, "servicelines_with_imputed_materials.csv")
    nbh_csv = os.path.join(DERIVED_DIR, "neighborhoods_from_matches_combined.csv")
    out_csv = os.path.join("data", "regression", "servicelines_lead_logits_dataset.csv")

    if not os.path.exists(houses_csv):
        raise SystemExit(f"Input not found: {houses_csv}")
    if not os.path.exists(nbh_csv):
        raise SystemExit(f"Input not found: {nbh_csv} (run build_neighborhood_from_matches.py)")

    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    houses = safe_read_csv(houses_csv)
    nbh = safe_read_csv(nbh_csv)

    # Resolve join keys
    nbh_key = pick_name_col(nbh)
    house_key = pick_name_col(houses)
    if nbh_key is None:
        raise SystemExit("Could not find a neighborhood name column in neighborhoods_from_matches_combined.csv")
    if house_key is None:
        raise SystemExit("Could not find a neighborhood name column in servicelines_with_imputed_materials.csv")

    # Prepare right-hand side columns with _neighborhood suffix
    nbh_renamed = suffix_columns(nbh.copy(), exclude=[nbh_key], suffix="_neighborhood")

    # Normalize keys for a robust merge (case/space insensitive)
    houses["__join_key__"] = normalize_key_series(houses[house_key])
    nbh_renamed["__join_key__"] = normalize_key_series(nbh[nbh_key])

    merged = houses.merge(
        nbh_renamed.drop(columns=[nbh_key]),
        on="__join_key__",
        how="left",
        suffixes=("", "_neighborhood"),
    )

    # Clean up temp key and, if useful, keep the original house neighborhood name
    merged = merged.drop(columns=["__join_key__"]).copy()

    merged.to_csv(out_csv, index=False)
    log(f"[OK] Wrote merged logits dataset -> {out_csv}")


if __name__ == "__main__":
    main()



[OK] Wrote merged logits dataset -> data\regression\servicelines_lead_logits_dataset.csv


In [1]:
# Imports + load minimalist EDA file
import pandas as pd
from pathlib import Path
import numpy as np
path = Path("data/regression/servicelines_lead_logits_dataset.csv")
df = pd.read_csv(path)

C:\Users\bradk\AppData\Local\Temp\ipykernel_31208\1326226116.py:6: DtypeWarning: Columns (31,32,33,34,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [3]:
from datetime import datetime

# drop missing years
df = df.dropna(subset=["yearstructbuilt"])
# drop years = "0"
df = df[df["yearstructbuilt"] > 0]
# drop future years
df = df[df["yearstructbuilt"] < 2026]
# bin by decade (or any bin width you like). 
df["year_bin"] = (df["yearstructbuilt"] // 10) * 10   # decade bins
#group all years before 1800
df.loc[df["yearstructbuilt"] < 1800, "year_bin"] = 1800
# Lead ban indicator: 0 = pre-1988, 1 = 1988 or later
df["post_lead_ban"] = (df["yearstructbuilt"] >= 1988).astype(int)
# Building age (in years, as of current year)
current_year = datetime.now().year
df["building_age"] = current_year - df["yearstructbuilt"]
df["building_age_decades"] = df["building_age"] // 10

# drop missing
df_model = df.dropna(subset=["bothsidesstatus_imputed", "Pblack", "householdincome_avghinc_cy", "year_bin"]).copy()

# scale income to 10k units for interpretability
df_model["MHI_10k"] = df_model["householdincome_avghinc_cy"] / 10000

# binary dependent variable
df_model["lead_flag"] = (df_model["bothsidesstatus_imputed"] == "Lead/GRR").astype(int)
df_model["cust_lead_flag"] = (df_model["custmaterial_cat_imputed"] == "Lead/GRR").astype(int)
df_model["util_lead_flag"] = (df_model["utilmaterial_cat_imputed"] == "Lead/GRR").astype(int)

# scale income to 10k units for interpretability
df_model["MHI_10k_neighborhood"] = df_model["householdincome_avghinc_cy_neighborhood"] / 10000


In [ ]:
# Gradient descent version:

# Standardize x
x = df_model["building_age"].values
y = df_model["lead_flag"].values
x_mean, x_std = x.mean(), x.std()
x_scaled = (x - x_mean) / x_std

alpha = 0.01
b0, b1 = 0.0, 0.0

for step in range(5000):
    db0, db1 = gradients(x_scaled, y, b0, b1)
    b0 -= alpha * db0
    b1 -= alpha * db1
    if step % 500 == 0:
        print(f"Step {step} | Loss={loss(x_scaled, y, b0, b1):.6f} | b0={b0:.6f} | b1={b1:.6f}")

# Convert back to original scale so we can compare to OLS
b1_unscaled = b1 / x_std
b0_unscaled = b0 - b1_unscaled * x_mean

print("\nRecovered (unscaled) coefficients:")
print(f"Intercept: {b0_unscaled:.6f} | Slope: {b1_unscaled:.6f}")


Step 0 | Loss=0.400320 | b0=0.008182 | b1=0.004689
Step 500 | Loss=0.186784 | b0=0.409108 | b1=0.234422
Step 1000 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 1500 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 2000 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 2500 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 3000 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 3500 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 4000 | Loss=0.186784 | b0=0.409125 | b1=0.234431
Step 4500 | Loss=0.186784 | b0=0.409125 | b1=0.234431

Recovered (unscaled) coefficients:
Intercept: -0.147971 | Slope: 0.007203


In [ ]:
# Linear regression (LPM version of the same model)


import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

formula = "lead_flag ~ building_age"
ols = smf.ols(formula=formula, data=df_model).fit()

print(ols.summary())

# Optional: show coefficient table in same style as your logit output
coef_table = pd.DataFrame({
    "term": ols.params.index,
    "coef": ols.params.values,
    "p_value": ols.pvalues,
})
coef_table["std_err"] = ols.bse
coef_table["t_stat"] = ols.tvalues
print("\nCoefficients:")
print(coef_table)


                            OLS Regression Results                            
Dep. Variable:              lead_flag   R-squared:                       0.227
Model:                            OLS   Adj. R-squared:                  0.227
Method:                 Least Squares   F-statistic:                 2.365e+04
Date:                Fri, 31 Oct 2025   Prob (F-statistic):               0.00
Time:                        13:21:51   Log-Likelihood:                -46620.
No. Observations:               80374   AIC:                         9.324e+04
Df Residuals:                   80372   BIC:                         9.326e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.1480      0.004    -37.648   